[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arrjon/BayesFlowTutorial/blob/main/01_Linear_Regression_Exercise.ipynb)

# BayesFlow 1 — From Pyro to Amortized Bayesian Inference

**Difficulty:** 🟢 Beginner

> 🎯 **What you will learn**
> - Revisit a Bayesian **linear regression** model in **Pyro** and solve one dataset with **NUTS**.
> - Reuse the *exact same* Pyro model as a **forward simulator** and train a **BayesFlow** amortized posterior estimator.
> - Confirm the BayesFlow posterior **matches** the Pyro/NUTS posterior on the same dataset.
> - See the payoff of **amortization**: instant posteriors for *many* datasets, where MCMC would rerun from scratch each time.

---

> ✏️ **How this notebook works.** Run each cell and look for **`TODO`** markers. Each one asks you to fill in a small piece of code.

In [ ]:
# ▶️  RUN THIS FIRST.  Installs this tutorial project and all its dependencies
#     (BayesFlow, JAX, Pyro, ...) from GitHub, as declared in pyproject.toml (~1-2 min).
import sys, subprocess

print("Installing the tutorial and its dependencies — this takes ~1-2 min ...")
_res = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/arrjon/BayesFlowTutorial.git"],
    capture_output=True, text=True,
)
if _res.returncode != 0:
    print(_res.stdout); print(_res.stderr)
    raise SystemExit("Install failed — see the pip output above.")

# NOTE: pip may internally warn that google-colab pins pandas==2.2.2 while BayesFlow
# needs pandas>=2.3.3. That warning is harmless and unavoidable (the tutorial doesn't use
# google-colab's pandas features).
print("✅ Setup complete.")
print("   If the NEXT cell raises a numpy/pandas import error, click "
      "Runtime ▸ Restart session and re-run.")

In [ ]:
import os
# BayesFlow runs on JAX, PyTorch or TensorFlow. We pick JAX here.
os.environ["KERAS_BACKEND"] = "jax"
import bayesflow as bf

> 📚 **Where to read more**
> - BayesFlow website & docs: <https://bayesflow.org>
> - Example gallery: <https://bayesflow.org/main/examples.html>
> - API reference: <https://bayesflow.org/main/api/bayesflow.html>

## From "one dataset at a time" to "all datasets at once"

Yesterday you used **Pyro** to do Bayesian inference: you wrote a probabilistic model and ran **MCMC (NUTS)** to approximate the posterior $p(\theta \mid x)$ for a dataset. That is exact and general — but it solves **one dataset at a time**. Got 500 datasets? Run NUTS 500 times.

**Amortized Bayesian inference** takes a different route. We train a neural network **once** to map *any* dataset $x$ directly to an approximate posterior $q_\phi(\theta \mid x)$. Training costs more up front, but afterwards inference for a new dataset is a single fast forward pass.

The neat part: we do not need access to the model, just simulations:
1. solve one dataset with **Pyro + NUTS**, and
2. hand the **same Pyro model** to **BayesFlow** as a data simulator and train an amortized estimator.

BayesFlow is organised into a few modules you'll meet below: `simulators` (draw batches), `adapters` (reshape for the networks), `networks` (summary + inference networks), and `workflows` (glue it together and train).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True)

# We use a single, fixed number of observations per dataset throughout this notebook.
N = 10

## 1. The model — in Pyro

Linear regression:

$$ \beta_0 \sim \mathrm{Normal}(2, 3), \quad \beta_1 \sim \mathrm{Normal}(0, 1), \quad \sigma \sim \mathrm{Gamma}(1, 1), \qquad x_i \sim \mathrm{Normal}(0,1), \quad y_i \sim \mathrm{Normal}(\beta_0 + \beta_1 x_i,\; \sigma). $$

We let the model generate the predictor $x$ **as well** as the response $y$. That way the single model function is a *complete* data generator. During inference we simply treat $x$ and $y$ as observed.

In [ ]:
import torch
import pyro
import pyro.distributions as dist
import pyro.poutine as poutine
from pyro.infer import MCMC, NUTS

def pyro_model(N, x=None, y=None):
    # priors
    beta = pyro.sample("beta", dist.Normal(torch.tensor([2., 0.]),
                                            torch.tensor([3., 1.])).to_event(1))
    sigma = pyro.sample("sigma", dist.Gamma(1., 1.))
    # data (x and y are sampled when not observed, i.e. when simulating)
    with pyro.plate("data", N):
        x = pyro.sample("x", dist.Normal(0., 1.), obs=x)
        y = pyro.sample("y", dist.Normal(beta[0] + beta[1] * x, sigma), obs=y)
    return x, y

Make **one** observed dataset with known ground-truth parameters (so we can check our answers later). We `condition` the model on chosen true values and run it forward — the model itself generates $x$ and $y$.

In [ ]:
true_beta = torch.tensor([2.0, 1.0])   # (intercept, slope)
true_sigma = torch.tensor(0.8)

pyro.set_rng_seed(7)
conditioned = poutine.condition(pyro_model, data={"beta": true_beta, "sigma": true_sigma})
trace = poutine.trace(conditioned).get_trace(N)
x_obs = trace.nodes["x"]["value"].numpy()
y_obs = trace.nodes["y"]["value"].numpy()

plt.scatter(x_obs, y_obs); plt.xlabel("x"); plt.ylabel("y"); plt.title("Our observed dataset");

## 2. Solve it with Pyro NUTS

Run MCMC on our one dataset, conditioning on the observed `x` and `y`. We time it for later comparison.

In [ ]:
import time
pyro.set_rng_seed(0)

t0 = time.time()
mcmc = MCMC(NUTS(pyro_model), num_samples=1000, warmup_steps=500)
mcmc.run(N, x=torch.tensor(x_obs).float(), y=torch.tensor(y_obs).float())
pyro_time = time.time() - t0

pyro_post = mcmc.get_samples()  # {'beta': (1000, 2), 'sigma': (1000,)}
print(f"NUTS finished in {pyro_time:.1f}s for ONE dataset")
print("posterior mean beta:", pyro_post["beta"].mean(0).numpy().round(2),
      " sigma:", float(pyro_post["sigma"].mean()))

> 🔎 That was fast for **one** small dataset. But NUTS starts over for every new dataset, and cost grows with data size and model complexity. If you had 500 datasets to fit, you'd pay that cost 500 times.

## 3. Hand the same model to BayesFlow

BayesFlow trains on **simulated** data: it needs a function that *generates* $(\theta, x, y)$ pairs. We already have one — our Pyro model! Running it forward (no observations) samples the parameters **and** the data. We wrap that in a tiny function and hand it to `make_simulator`.

We could also write the same simulator with simple numpy functions, as we are not differentiating through it. But using the Pyro model is convenient and ensures the simulator matches the model exactly.

In [ ]:
def simulator_fn(num_obs=N):
    """Run the Pyro model forward once and return one simulated dataset."""
    tr = poutine.trace(pyro_model).get_trace(num_obs)
    return dict(
        beta=tr.nodes["beta"]["value"].numpy(),
        sigma=float(tr.nodes["sigma"]["value"]),
        x=tr.nodes["x"]["value"].numpy(),
        y=tr.nodes["y"]["value"].numpy(),
    )

simulator = bf.make_simulator([simulator_fn])
sim_draws = simulator.sample(500)
print("beta:", sim_draws["beta"].shape, " x:", sim_draws["x"].shape)

### The Adapter

Neural networks want clean, standardized tensors and must be told which variables are **targets** and which are **data**. We keep `sigma` **positive**, and stack variables into the two keys BayesFlow expects: `inference_variables` (targets) and `summary_variables` (data).

In [ ]:
adapter = (
    bf.Adapter()
    .as_set(["x", "y"])
    .constrain("sigma", lower=0)
    .convert_dtype("float64", "float32")
    .concatenate(["beta", "sigma"], into="inference_variables")
    .concatenate(["x", "y"], into="summary_variables")
)

processed = adapter(sim_draws)
print(processed["summary_variables"].shape, processed["inference_variables"].shape)  # (500, N, 2) (500, 3)

### Networks

Two networks work together:
- A **summary network** compresses each dataset (the `N` observations) into a fixed-length embedding.
- An **inference network** (`CouplingFlow`, a normalizing flow) turns that embedding into posterior draws. We provide this one for you.

The interesting decision is the **summary network** — that's your task below. Think about the *structure* of our data before you choose.

In [ ]:
# TODO(1): pick the right SUMMARY network for this problem.
#
#   Key question: what is the STRUCTURE of one dataset here?
#   The summary network you pick must respect that.
#
#   Uncomment the ONE that is appropriate:
#
#   summary_network = bf.networks.DeepSet(summary_dim=10)
#   summary_network = bf.networks.TimeSeriesNetwork(summary_dim=10)
#
summary_network = ...  # <-- replace with your choice

# Rule of thumb: summary_dim should be at least 2x the number of parameters.

In [ ]:
# Inference network: a coupling flow.
inference_network = bf.networks.CouplingFlow(transform="spline")

workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_network,
    summary_network=summary_network,
)

### Train (once!)

We could simulate a fixed training set and train **offline**. Or we could train **online** by simulating a new batch of datasets every step. The latter is more robust to overfitting, and should be done when the simulator is cheap (On JAX the first steps compile and are slower; that's normal. Exepect it to run 1-2 minutes on a Colab CPU, seconds on a GPU).

In [ ]:
history = workflow.fit_online(epochs=50, batch_size=128, num_batches_per_epoch=10)
f = bf.diagnostics.plots.loss(history)

## 4. Do the two posteriors agree? BayesFlow vs Pyro

The real test: run BayesFlow on the **exact same dataset** we gave to NUTS, and overlay the two posteriors. If BayesFlow learned the right thing, the sample clouds should sit on top of each other.

For inference on a specific dataset we just pass its `x` and `y` as `conditions` — the adapter applies the same transforms it used in training.

In [ ]:
def compare_posteriors(pyro_post, bf_post, truth):
    names = [r"$\beta_0$", r"$\beta_1$", r"$\sigma$"]
    pyro_arr = np.column_stack([pyro_post["beta"].numpy(), pyro_post["sigma"].numpy()[:, None]])
    bf_arr = np.column_stack([bf_post["beta"][0], bf_post["sigma"][0]])
    fig, ax = plt.subplots(1, 3, figsize=(12, 3), layout="constrained")
    for i, a in enumerate(ax):
        a.hist(pyro_arr[:, i], bins=40, density=True, alpha=0.5, color="#4C72B0", label="Pyro (NUTS)")
        a.hist(bf_arr[:, i], bins=40, density=True, alpha=0.5, color="#E7298A", label="BayesFlow")
        a.axvline(truth[i], color="black", ls="--", lw=1, label="truth")
        a.set_xlabel(names[i])
    ax[0].set_ylabel("density"); ax[0].legend()
    return fig

In [ ]:
obs_dataset = {
    "x": x_obs[None, :],
    "y": y_obs[None, :],
    "num_obs": N
}
bf_post = workflow.sample(conditions=obs_dataset, num_samples=1000)

compare_posteriors(pyro_post, bf_post,
                   truth=[float(true_beta[0]), float(true_beta[1]), float(true_sigma)]);

> 🔎 The two posteriors should overlap closely. Same answer, very different route to get there.

## 5. The payoff: amortization

Here's what the extra training bought us. Pyro solved **one** dataset. BayesFlow can now solve **any number** of datasets, each in a fast forward pass — no retraining, no re-running MCMC.

In [ ]:
import time
many = simulator.sample(200)  # 200 brand-new datasets

t0 = time.time()
post_many = workflow.sample(conditions=many, num_samples=1000)
bf_time = time.time() - t0

print(f"BayesFlow: posteriors for 200 datasets in {bf_time:.1f}s total.")
print(f"Pyro/NUTS: ~{pyro_time:.1f}s PER dataset  ->  ~{pyro_time * 200:.0f}s for 200 datasets"
      " (and that's the easy case).")

In [ ]:
par_names = [r"$\beta_0$", r"$\beta_1$", r"$\sigma$"]
f = bf.diagnostics.plots.recovery(estimates=post_many, targets=many, variable_names=par_names)

> 💡 **This is amortization.** The training cost is paid **once**; inference is then near-free and reusable. When you need posteriors for many datasets — or want instant re-analysis as new data arrives — amortized inference wins. When you only ever have one dataset, classic MCMC like Pyro's NUTS is often the simpler choice, but only if you can evalute the likelihood of the model. BayesFlow on the other hand uses only simulations, so it can handle models where the likelihood is intractable or unknown.

## 🎉 Nicely done!

You defined a model **once in Pyro**, solved it with **NUTS**, reused the *same model* as a **BayesFlow** simulator, confirmed the posteriors agree, and saw why amortization matters.

**What next?**
1. How to amortize over different dataset sizes (varying `N`)? Try retraining with the simulator below
2. You can also use more performant summary networks (e.g. `SetTransformer`), and explore the effect of hyperparameters on training speed and posterior quality.
3. How would you *validate* the amortized posterior when you have **no** ground truth and **no** MCMC reference? (That is exactly what **simulation-based calibration** is for — the topic of the next notebook.)

## 6. Additional Exercise: amortize over varying dataset sizes

Observe that the simulator above always generates datasets of size `N`. If you want to amortize over **different dataset sizes**, you can modify the simulator to draw a random number of observations each time. The adapter must then broadcast that number to the data arrays, and the summary network must be permutation-invariant (as it already is). We also pass the number of observations to the adapter so it can be used as an inference condition and the inference network knows how many datasets were passed. Use the below simulator and adapter above to train a new BayesFlow workflow that can handle datasets of varying sizes. Now you should increase the training budget, e.g., `num_batches_per_epoch=100`.

In [ ]:
def meta():
    # number of observation in a dataset
    num_obs = np.random.randint(5, 15)
    return dict(num_obs=num_obs)

simulator = bf.simulators.make_simulator([simulator_fn], meta_fn=meta)

adapter = (
    bf.Adapter()
    .broadcast("num_obs", to="x")
    .as_set(["x", "y"])
    .sqrt("num_obs")
    .constrain("sigma", lower=0)
    .convert_dtype("float64", "float32")
    .concatenate(["beta", "sigma"], into="inference_variables")
    .concatenate(["x", "y"], into="summary_variables")
    .rename("num_obs", "inference_conditions")
)

In [ ]:
# TODO(2): train a new BayesFlow workflow that can handle datasets of varying sizes. Then run the code below to check the posterior contraction as a function of the number of datasets.

In [ ]:
post_data_all = []
data_all = []
contraction_all = []
for n in range(5, 15):
    data_n = simulator.sample(100, num_obs=n)
    post_data_n = workflow.sample(conditions=data_n, num_samples=1000)
    post_data_n = np.concatenate([post_data_n['beta'], post_data_n['sigma']], axis=-1)
    data_n_target = np.concatenate([data_n['beta'], data_n['sigma']], axis=-1)
    post_data_all.append(post_data_n)
    data_all.append(data_n_target)

    contraction_n = bf.diagnostics.posterior_contraction(post_data_n, data_n_target, variable_names=par_names)
    contraction_all.append(contraction_n['values'])

post_data_all = np.concatenate(post_data_all, axis=0)
data_all = np.concatenate(data_all, axis=0)
f = bf.diagnostics.plots.recovery(estimates=post_data_all, targets=data_all, variable_names=par_names)

plt.plot(np.arange(5,15), [c[0] for c in contraction_all], label=par_names[0])
plt.plot(np.arange(5,15), [c[1] for c in contraction_all], label=par_names[1])
plt.plot(np.arange(5,15), [c[2] for c in contraction_all], label=par_names[2])
plt.xlabel("Number of Datasets")
plt.ylabel("Posterior Contraction")
plt.legend()
plt.show()